# Data Collection and Processing

Turns raw per-run simulation output (`data/outputs/runs/`) into the two
aggregated tables for later use:

- **`agent_level_data.parquet`**:  one row per agent per run, with the three
  agent-level belief-dynamics metrics. The include **plasticity**, **directedness** (`monotonicity` in code), and **outgoing influence**.
- **`run_level_data.parquet`** — one row per run, with the population-level
  **consensus** outcome used to build *consensus change*.

Only the metrics that are actually reported in the paper are kept here; the
raw simulation logs contain a few additional exploratory quantities (e.g.
label entropy, "polarization", per-class plasticity breakdowns) that were not
used for any reported result and have been dropped to keep this notebook
focused.


In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl

sys.path.insert(0, str(Path.cwd().parent))
from src.analysis.loaders import load_adjacency_matrix, load_agents_data, load_run_data, load_runs_metadata


## Role <-> model matching

Scenario IV ("Specialists with Matching Roles") assigns each specialist LLM a
social role that matches its finetuning domain (e.g. the chemistry model gets
the "Chemist" role; `Appendix D`). `MATCHES` records that mapping so we can
label each agent as `expert_matched` vs. `expert_mismatched` vs. `base` below.


In [ ]:
MATCHES = [
    ("llama-doc", "Clinical Physician"),
    ("llama-base", "LLM"),
    ("llama-base", "Human Participant"),
    ("llama-assistant", "Assistant"),
    ("llama-biomed", "Biomedical Researcher"),
    ("llama-chemist", "Chemist"),
    ("llama-coder", "Software Engineer"),
    ("llama-cyber", "Cybersecurity Analyst"),
    ("llama-finance", "Financial Analyst"),
    ("llama-hermes", "Strategic Planner"),
    ("llama-openmath", "Mathematician"),
    ("llama-lexicographer", "Lexicographer"),
    ("llama-linguist", "Language Mediator"),
    ("llama-roleplay", "Storyteller"),
    ("llama-scholar", "Academic Scholar"),
    ("llama-user", "Online Friend"),
]

SETTINGS = ["base_llms", "random_roles", "random_experts", "experts"]


## Load run metadata

In [ ]:
df_total = pl.from_pandas(load_runs_metadata(Path.cwd().parent / "data" / "outputs" / "runs"))

# Only runs that finished and passed post-hoc validation (see sanity_check.ipynb).
df_runs = df_total.filter(pl.col("validated") & pl.col("completed")).to_pandas()

print(f"Total validated runs: {len(df_runs)}")
print("\nRuns by setting:")
print(df_runs.groupby("setting").size())
print("\nRuns by graph type:")
print(df_runs.groupby("graph_type").size())

result_path = Path.cwd().parent / "data" / "outputs" / "aggregated"
result_path.mkdir(parents=True, exist_ok=True)


In [ ]:
print("Runs per (setting, statement) - should be balanced across statements:")
df_runs.groupby(["setting", "statement_id"]).agg("size").unstack(fill_value=0)


## 1. Agent-level metrics

For each agent in each run, we track its belief trajectory
`b_i^(t) = [P(correct), P(incorrect), P(neither)]` across `T=10` rounds and
compute:

- **Plasticity** (Eq. 3): average per-round magnitude of belief change.
- **Directedness** (Eq. 4, `monotonicity` below): whether that movement is
  consistently in one direction (-> 1) or oscillates (-> 0). Undefined
  (`NaN`) when plasticity is exactly zero (an agent that never moves has no
  "direction" to speak of) — see the NaN-rate check at the end of this section.


In [ ]:
def compute_agent_stats(belief_traj: np.ndarray) -> dict:
    """Plasticity (Eq. 3) and directedness (Eq. 4) for one agent's belief trajectory.

    Args:
        belief_traj: Array of shape (T+1, 3), the agent's [P(correct), P(incorrect),
            P(neither)] at each round 0..T.

    Returns:
        dict with `plasticity_tv`, `monotonicity` (directedness), and `T`.
    """
    P = np.asarray(belief_traj, dtype=float)  # (T+1, 3)
    T = P.shape[0] - 1

    # Per-round total-variation distance: 0.5 * ||b(t+1) - b(t)||_1
    diffs = np.abs(np.diff(P, axis=0))  # (T, 3)
    plasticity_tv = 0.5 * diffs.sum(axis=1).mean()

    # Directedness (Eq. 4): net displacement over T rounds, relative to total
    # movement accumulated along the way. 1 = monotone movement, 0 = oscillation.
    net_change = 0.5 * np.abs(P[-1] - P[0]).sum()
    denom = T * plasticity_tv
    monotonicity = float(net_change / denom) if denom > 1e-12 else np.nan

    return {"plasticity_tv": float(plasticity_tv), "monotonicity": monotonicity, "T": T}


## 2. Outgoing influence 

`Outgoing Influence(i)` measures whether agent `i`'s belief updates are followed by
larger subsequent updates among its neighbors: it weights each neighbor's
next-round movement `delta_j(t+1)` by how much agent `i` itself moved at `t`,
then averages over neighbors and rounds.


In [ ]:
def compute_outgoing_influence(belief_traj_all: np.ndarray, A: np.ndarray) -> np.ndarray:
    """Outgoing influence (Eq. 6) for every agent in a run.

    Args:
        belief_traj_all: Array of shape (N, T+1, 3), all agents' belief trajectories.
        A: (N, N) binary adjacency matrix.

    Returns:
        (N,) array: Influence(i) for each agent i.
    """
    N, Tp1, _ = belief_traj_all.shape
    T = Tp1 - 1
    if T < 2:
        raise ValueError(f"Need at least 2 transitions for influence; got T={T}.")

    delta = np.diff(belief_traj_all, axis=1)          # (N, T, 3)
    tv = 0.5 * np.abs(delta).sum(axis=2)               # (N, T) per-round TV movement

    influence = np.zeros(N)
    eps = 1e-12
    for i in range(N):
        neighbors = np.where(A[i] > 0)[0]
        if len(neighbors) == 0:
            continue

        delta_i_own_move = tv[i, :-1]        # own movement at rounds 0..T-2, i.e. delta_i^(t)
        delta_j_next_move = tv[neighbors, 1:]  # neighbors' movement at rounds 1..T-1, i.e. delta_j^(t+1)

        numerator = (delta_i_own_move[None] * delta_j_next_move).sum()  # sum_{j,t} delta_i(t) * delta_j(t+1)
        denominator = delta_i_own_move.sum() * len(neighbors)           # |N(i)| * sum_t delta_i(t)

        influence[i] = numerator / denominator if denominator > eps else 0.0

    return influence


## 3. Build the agent-level table

In [ ]:
records = []
lf = pl.from_pandas(df_runs)

for run in lf.filter(pl.col("setting").is_in(SETTINGS)).iter_rows(named=True):
    run_agents = load_agents_data(run["run_path"])
    graph_data = load_run_data(run["run_path"])
    A, _ = load_adjacency_matrix(run["run_path"])

    belief_ful = run_agents["belief_ful"]  # (N_agents, T+1, 3): [P(correct), P(incorrect), P(neither)]
    models, roles = run_agents["models"], run_agents["roles"]

    outgoing_influence = compute_outgoing_influence(belief_ful, A)
    try:
        agent_ids = [v["node_id"] for v in graph_data["agents_data"].values()]
    except KeyError:
        agent_ids = [int(k) for k in graph_data["agents_data"].keys()]

    for i in range(belief_ful.shape[0]):
        stats = compute_agent_stats(belief_ful[i])
        is_matched = (models[int(i)], roles[int(i)]) in MATCHES
        is_expert = not models[int(i)].startswith("llama-base")

        records.append({
            "agent_id": agent_ids[i],
            "plasticity_tv": stats["plasticity_tv"],
            "monotonicity": stats["monotonicity"],
            "influence_out_joint": outgoing_influence[int(i)],
            "T": stats["T"],
            "is_matched": is_matched,
            "is_expert": is_expert,
            "role": roles[int(i)],
            "model": models[int(i)],
            "statement_id": run["statement_id"],
            "setting": run["setting"],
            "seed": run["seed"],
            "graph_seed": str(graph_data["config"]["network"]["seed_map"][str(run["seed"])]),
            "graph_id": f"{run['graph_type']}-{run['seed']}",
            "graph_type": run["graph_type"],
        })

dft = pd.DataFrame(records)

dft["setting"] = pd.Categorical(dft["setting"], categories=SETTINGS)
for col in ["model", "graph_type", "role", "statement_id", "graph_seed"]:
    dft[col] = pd.Categorical(dft[col], categories=sorted(dft[col].astype(str).unique()))

# Agent identity relative to Scenario IV's role-matching design (Appendix D):
# a generalist ("base"), a specialist in a randomly-assigned ("mismatched") role,
# or a specialist whose role matches its finetuning domain ("matched").
dft["agent_configs"] = np.select(
    [~dft["is_expert"], dft["is_expert"] & ~dft["is_matched"], dft["is_expert"] & dft["is_matched"]],
    ["base", "expert_mismatched", "expert_matched"],
    default="base",
)

dft["run_id"] = (
    dft["setting"].astype(str) + ":" + dft["graph_id"].astype(str) + ":" + dft["statement_id"].astype(str)
)
dft["run_id"] = pd.Categorical(dft["run_id"], categories=sorted(dft["run_id"].unique()))

dft.to_parquet(result_path / "agent_level_data.parquet", index=False)
dft.head()


### NaN check on directedness

`monotonicity` (directedness) is undefined when an agent's plasticity is
exactly zero (no movement to have a "direction"). This confirms how rare that
is, rather than silently carrying NaNs into the mixed-effects models in
`X2_agent_analysis.ipynb` (which drop them by default).


In [ ]:
if "dft" not in globals():
    dft = pd.read_parquet(result_path / "agent_level_data.parquet")

n_nan = dft["monotonicity"].isna().sum()
pct_nan = dft["monotonicity"].isna().mean() * 100
print(f"NaNs in directedness (monotonicity): {n_nan} / {len(dft)} ({pct_nan:.2f}%)")

dft.loc[dft["monotonicity"].isna()].sample(min(10, n_nan), random_state=0) if n_nan > 0 else dft.sample(10, random_state=0)


## 4. Population-level metric: consensus

For each run, we track the discretized label distribution across all `n=48`
agents and compute **consensus** as the modal-label share (fraction of agents
agreeing with the majority label) at round 0 and round `T`.


In [ ]:
def compute_run_consensus(belief_traj_all: np.ndarray) -> dict:
    """Modal-label consensus (Fig. 2C's operationalization) at round 0 and round T.

    Consensus is about *agreement among agents*, not correctness, so this
    needs only the discretized beliefs - not the statement's ground truth.

    Args:
        belief_traj_all: (N, T+1, 3) belief trajectories for all agents in a run.

    Returns:
        dict with `modal_consensus_0`, `modal_consensus_T`, `T`.
    """
    P = np.asarray(belief_traj_all, dtype=float)  # (N, T+1, 3)
    N, Tp1, K = P.shape
    T = Tp1 - 1

    labels = P.argmax(axis=2)  # (N, T+1) discretized belief per agent per round

    def modal_share(labels_t: np.ndarray) -> float:
        counts = np.bincount(labels_t, minlength=K)
        return counts.max() / N

    return {
        "modal_consensus_0": float(modal_share(labels[:, 0])),
        "modal_consensus_T": float(modal_share(labels[:, T])),
        "T": T,
    }


In [ ]:
records = []

for run in lf.filter(pl.col("setting").is_in(SETTINGS)).iter_rows(named=True):
    run_agents = load_agents_data(run["run_path"])
    run_data = load_run_data(run["run_path"])

    belief_ful = run_agents["belief_ful"]
    consensus = compute_run_consensus(belief_ful)

    records.append({
        "T": consensus["T"],
        "statement_id": run["statement_id"],
        "setting": run["setting"],
        "seed": run["seed"],
        "graph_seed": str(run_data["config"]["network"]["seed_map"][str(run["seed"])]),
        "graph_id": f"{run['graph_type']}-{run['seed']}",
        "graph_type": run["graph_type"],
        "modal_consensus_0": consensus["modal_consensus_0"],
        "modal_consensus_T": consensus["modal_consensus_T"],
        "modal_consensus_change": consensus["modal_consensus_T"] - consensus["modal_consensus_0"],
    })

dft_runs = pd.DataFrame(records)

dft_runs["setting"] = pd.Categorical(dft_runs["setting"], categories=SETTINGS)
for col in ["graph_type", "statement_id", "graph_seed"]:
    dft_runs[col] = pd.Categorical(dft_runs[col], categories=sorted(dft_runs[col].astype(str).unique()))

dft_runs["run_id"] = (
    dft_runs["setting"].astype(str) + ":" + dft_runs["graph_id"].astype(str) + ":" + dft_runs["statement_id"].astype(str)
)
dft_runs["run_id"] = pd.Categorical(dft_runs["run_id"], categories=sorted(dft_runs["run_id"].unique()))

dft_runs.to_parquet(result_path / "run_level_data.parquet", index=False)
dft_runs.head()
